In [5]:
# Install dependencies in Colab
!pip install bloom-filter ecdsa base58 pycryptodome tqdm psutil -q

import hashlib
import base58
import ecdsa
import binascii
import time
import multiprocessing as mp
from bloom_filter import BloomFilter
from Crypto.Hash import RIPEMD160
from tqdm import tqdm
from multiprocessing import Queue  # Queue only
from multiprocessing.queues import Empty  # Correct Empty import
import psutil

# Known values
TARGET_X = "7a51392bace353f4c3788c9c090ef4f635ec211159ec3b9f1bb7da7679517e12"
TARGET_Y = "6e98e0012bcb4d2b023c479afaaa1ad703ea1b24e1910e2cdad38744ba7aab8a"
TARGET_HASH160 = "31f19a7d0379f56cb3be0761c21f1f0c9553a47f"
TARGET_WIF_PREFIX = "5"

G = ecdsa.SECP256k1.generator

def generate_private_key(seed):
    seed_str = str(seed).encode('utf-8')
    priv_key = hashlib.sha256(seed_str).digest()[:6]
    priv_key = b'\x00' * 26 + priv_key
    return priv_key

def private_to_wif_and_checksum(priv_key):
    extended = b'\x80' + priv_key
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    wif = base58.b58encode(extended + checksum).decode('utf-8')
    return wif, checksum.hex()

def private_to_public_key(priv_key):
    sk = ecdsa.SigningKey.from_string(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.verifying_key
    pub_key = b'\x04' + vk.to_string()
    x, y = pub_key[1:33].hex(), pub_key[33:].hex()
    sha = hashlib.sha256(pub_key).digest()
    ripemd = RIPEMD160.new()
    ripemd.update(sha)
    hash160 = ripemd.digest().hex()
    return x, y, hash160

def sanity_check(priv_key, wif, checksum, x, y, hash160):
    return (wif.startswith(TARGET_WIF_PREFIX) and
            x == TARGET_X and
            y == TARGET_Y and
            hash160 == TARGET_HASH160)

def process_seeds(args):
    seed_range, bloom, queue, prefix_counts = args
    batch_size = 100000  # ~3.2 MB/batch, safe with 51 GB

    for i in range(0, len(seed_range), batch_size // 1000):
        batch = seed_range[i:i + batch_size // 1000]
        for t in batch:
            for pid in range(1000):
                seed = f"{t}.{pid}"
                priv_key = generate_private_key(seed)
                wif, checksum = private_to_wif_and_checksum(priv_key)
                prefix = wif[:2]
                prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1
                queue.put((checksum, priv_key, wif, seed))

def process_checksum_chunk(chunk_queue, bloom, result_queue):
    bloom_hits = 0
    while True:
        try:
            item = chunk_queue.get(timeout=1)
            checksum, priv_key, wif, seed = item
            x, y, hash160 = private_to_public_key(priv_key)
            if hash160 in bloom and y in bloom:
                bloom_hits += 1
                if x == TARGET_X:
                    if sanity_check(priv_key, wif, checksum, x, y, hash160):
                        result_queue.put((priv_key.hex(), wif, checksum, seed, bloom_hits))
                        chunk_queue.task_done()
                        return
            chunk_queue.task_done()
        except Empty:
            break
    result_queue.put((None, None, None, None, bloom_hits))

def main():
    bloom = BloomFilter(max_elements=2, error_rate=0.0001)
    bloom.add(TARGET_HASH160)
    bloom.add(TARGET_Y)

    start_time = 1231006505
    end_time = 1270915355
    total_seconds = (end_time - start_time) // 1000
    chunk_size = total_seconds // 8

    seed_ranges = [
        range(start_time + i * chunk_size * 1000,
              start_time + (i + 1) * chunk_size * 1000 if i < 7 else end_time,
              1000)
        for i in range(8)
    ]

    total_seeds = sum(len(r) for r in seed_ranges) * 1000
    print(f"Scanning {total_seeds} seeds (~2^{int(total_seeds).bit_length()}) with Bloom filter (FPR=0.0001) across 8 cores...")
    start = time.time()

    # Manager for shared objects
    manager = mp.Manager()
    seed_queue = manager.Queue(maxsize=10000)  # ~320 MB max
    result_queue = manager.Queue()
    prefix_counts = manager.dict()

    # Start mapping processes first
    map_processes = []
    for _ in range(8):
        p = mp.Process(target=process_checksum_chunk, args=(seed_queue, bloom, result_queue))
        map_processes.append(p)
        p.start()

    # Seed generation (parallel)
    gen_processes = []
    with tqdm(total=total_seeds, desc="Generation Progress") as pbar:
        for seed_range in seed_ranges:
            p = mp.Process(target=process_seeds, args=(seed_range, bloom, seed_queue, prefix_counts))
            gen_processes.append(p)
            p.start()

        processed = 0
        while any(p.is_alive() for p in gen_processes + map_processes) or not seed_queue.empty():
            while not seed_queue.empty():
                seed_queue.get()
                processed += 1
                pbar.update(1)
            time.sleep(0.1)

        for p in gen_processes + map_processes:
            p.join()

    # Aggregate results
    total_bloom_hits = 0
    for _ in range(8):
        priv_key_hex, wif, checksum, seed, bloom_hits = result_queue.get()
        total_bloom_hits += bloom_hits
        if priv_key_hex:
            print(f"\nX match found! Seed={seed}, WIF={wif}, Checksum={checksum}")
            print(f"Sanity check passed: priv_key={priv_key_hex}, WIF={wif}, Checksum={checksum}")
            print(f"Private key found: {priv_key_hex} -> WIF: {wif}")
            print(f"Time taken: {time.time() - start:.2f} seconds")
            break
    else:
        print(f"\nNo match found. Time taken: {time.time() - start:.2f} seconds")

    print(f"Total Bloom hits: {total_bloom_hits}")
    print("\nPrefix distribution (top 5):")
    for prefix, count in sorted(prefix_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{prefix}: {count} occurrences")

if __name__ == "__main__":
    main()

Scanning 39909000 seeds (~2^26) with Bloom filter (FPR=0.0001) across 8 cores...


Generation Progress:   0%|          | 0/39909000 [00:00<?, ?it/s]Process Process-53:
Process Process-54:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process Process-55:
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
TypeError: process_seeds() takes 1 positional argument but 4 were given
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
TypeError: process_seeds() takes 1 positional argument but 4 were given
Process Process-56:
  File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
    


No match found. Time taken: 1.12 seconds
Total Bloom hits: 0

Prefix distribution (top 5):
